In [5]:
import pandas as pd
import numpy as np

# =============================================================================
# CONFIGURATION
# =============================================================================
# Replace this with the actual name of your Top 10 CSV file
INPUT_FILE = '20_gene_to_mz_results_v1_analytic_fast/gene_to_mz_top30_matches_all_scores.csv' 

# RRF Constant (k): 
# k=0 makes Rank 1 much more valuable than Rank 2 (aggressive).
# k=60 makes Rank 1 and Rank 10 closer in value (smoother).
# For Top 10 lists, k=1 is usually a good balance.
K_FACTOR = 100

# Optional: Filter by Group (set to None to use all samples)
# You can change this to 'AAD' or 'YC' to see if matches differ by condition.
FILTER_GROUP = None  


In [ ]:

# =============================================================================
# CONSENSUS ALGORITHM
# =============================================================================

def calculate_consensus(df, gene_name):
    """Calculates the RRF score for a specific gene."""
    subset = df[df['Gene'] == gene_name]
    
    if subset.empty:
        return None
    
    # Trackers
    rrf_scores = {}       # {mz: score}
    sample_counts = {}    # {mz: number_of_samples_found}
    best_ranks = {}       # {mz: best_rank_achieved}
    
    total_samples = subset['Sample'].nunique()
    
    for _, row in subset.iterrows():
        mz = row['MZ_Feature']
        rank = row['Rank']
        
        # Initialize
        if mz not in rrf_scores:
            rrf_scores[mz] = 0.0
            sample_counts[mz] = 0
            best_ranks[mz] = 99
        
        # RRF Formula: Score += 1 / (Rank + k)
        rrf_scores[mz] += 1.0 / (rank + K_FACTOR)
        
        sample_counts[mz] += 1
        if rank < best_ranks[mz]:
            best_ranks[mz] = rank
            
    if not rrf_scores:
        return None

    # Sort candidates by Score (Highest = Best)
    sorted_candidates = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    # Get the Winner
    winner_mz, winner_score = sorted_candidates[0]
    
    return {
        'winner': winner_mz,
        'score': winner_score,
        'count': sample_counts[winner_mz],
        'total_samples': total_samples,
        'best_rank': best_ranks[winner_mz],
        'runner_up': sorted_candidates[1] if len(sorted_candidates) > 1 else None
    }

# =============================================================================
# MAIN EXECUTION
# =============================================================================

# Load Data
try:
    df = pd.read_csv(INPUT_FILE)
    # Ensure column names match standard format
    df.columns = [c.title() for c in df.columns] # Enforce Capitalized Titles (Gene, Sample, Rank...)
    if 'Rna_Sample' in df.columns: df.rename(columns={'Rna_Sample': 'Sample'}, inplace=True)
    if 'Mz_Feature' in df.columns: df.rename(columns={'Mz_Feature': 'MZ_Feature'}, inplace=True)
    
except FileNotFoundError:
    print(f"Error: Could not find '{INPUT_FILE}'. Make sure the file exists.")
    exit()

# Filter by Group if requested
if FILTER_GROUP:
    print(f"Filtering for samples containing '{FILTER_GROUP}'...")
    df = df[df['Sample'].str.contains(FILTER_GROUP)]

unique_genes = df['Gene'].unique()

print(f"\n{'GENE':<15} | {'CONSENSUS MATCH (m/z)':<25} | {'STABILITY':<15} | {'CONFIDENCE'}")
print("-" * 85)

results = []

for gene in unique_genes:
    res = calculate_consensus(df, gene)
    if not res: continue
    
    winner = str(res['winner'])
    freq = f"{res['count']}/{res['total_samples']}"
    pct = (res['count'] / res['total_samples']) * 100
    
    # Determine Confidence Level
    confidence = "LOW"
    if pct >= 50: confidence = "MEDIUM"
    if pct >= 75: confidence = "HIGH"
    if pct >= 90: confidence = "VERY HIGH"
    
    print(f"{gene:<15} | {winner:<25} | {freq:<15} | {confidence}")
    
    # Optional detailed output for ambiguity
    if confidence == "LOW" and res['runner_up']:
        run_mz, run_score = res['runner_up']
        print(f"   ↳ Ambiguous. Runner-up: {run_mz} (Score: {run_score:.2f} vs Winner: {res['score']:.2f})")

    results.append({
        'Gene': gene,
        'Consensus_MZ': winner,
        'Stability_Count': res['count'],
        'Total_Samples': res['total_samples'],
        'Confidence': confidence
    })

# Save Final Consensus List
pd.DataFrame(results).to_csv('gene_to_mz_results_v1_analytic_fast/final_gene_mz_consensus_30.csv', index=False)
print("-" * 85)
print("Consensus list saved to 'final_gene_mz_consensus.csv'")


GENE            | CONSENSUS MATCH (m/z)     | STABILITY       | CONFIDENCE
-------------------------------------------------------------------------------------
AC149090.1      | 770.507                   | 5/16            | LOW
   ↳ Ambiguous. Runner-up: 846.5419 (Score: 0.04 vs Winner: 0.05)
Aplp1           | 719.5768                  | 6/16            | LOW
   ↳ Ambiguous. Runner-up: 780.5501 (Score: 0.05 vs Winner: 0.05)
Apoe            | 719.5768                  | 4/16            | LOW
   ↳ Ambiguous. Runner-up: 768.587 (Score: 0.04 vs Winner: 0.04)
App             | 754.5364                  | 4/16            | LOW
   ↳ Ambiguous. Runner-up: 781.5576 (Score: 0.04 vs Winner: 0.04)
Atp1a3          | 630.6164                  | 5/16            | LOW
   ↳ Ambiguous. Runner-up: 772.6211 (Score: 0.05 vs Winner: 0.05)
Elavl3          | 772.6211                  | 5/16            | LOW
   ↳ Ambiguous. Runner-up: 843.6645 (Score: 0.04 vs Winner: 0.04)
Eno1            | 844.6774         

In [7]:
import pandas as pd

# CONFIGURATION
INPUT_FILE = '20_gene_to_mz_results_v1_analytic_fast/gene_to_mz_top30_matches_all_scores.csv' 
GROUPS = ['YC', 'YAD', 'AC', 'AAD']  # The prefixes of your sample names
K_FACTOR = 1

def calculate_group_consensus(df, gene_name, group_prefix):
    # Filter for samples in this group
    subset = df[(df['Gene'] == gene_name) & (df['Sample'].str.startswith(group_prefix))]
    
    if subset.empty: return None, 0, 0

    rrf_scores = {}
    counts = {}
    total_samples = subset['Sample'].nunique()

    for _, row in subset.iterrows():
        mz = row['MZ_Feature']
        rank = row['Rank']
        
        if mz not in rrf_scores:
            rrf_scores[mz] = 0.0
            counts[mz] = 0
            
        rrf_scores[mz] += 1.0 / (rank + K_FACTOR)
        counts[mz] += 1
    
    if not rrf_scores: return None, 0, total_samples

    # Find winner
    winner_mz = max(rrf_scores, key=rrf_scores.get)
    return winner_mz, counts[winner_mz], total_samples

# Load Data
try:
    df = pd.read_csv(INPUT_FILE)
    # Normalize columns
    df.columns = [c.title() for c in df.columns] 
    if 'Rna_Sample' in df.columns: df.rename(columns={'Rna_Sample': 'Sample'}, inplace=True)
    if 'Mz_Feature' in df.columns: df.rename(columns={'Mz_Feature': 'MZ_Feature'}, inplace=True)
except FileNotFoundError:
    print("File not found.")
    exit()

unique_genes = df['Gene'].unique()

print(f"{'GENE':<10} | {'GROUP':<5} | {'CONSENSUS MATCH (m/z)':<20} | {'STABILITY':<10} | {'STATUS'}")
print("-" * 75)

for gene in unique_genes:
    print(f"--- {gene} ---")
    
    for group in GROUPS:
        winner, count, total = calculate_group_consensus(df, gene, group)
        
        if winner:
            pct = (count / total) * 100
            status = "WEAK"
            if pct >= 50: status = "MODERATE"
            if pct >= 75: status = "STRONG"
            if pct == 100: status = "PERFECT"
            
            print(f"{'':<10} | {group:<5} | {str(winner):<20} | {count}/{total}       | {status}")
        else:
            print(f"{'':<10} | {group:<5} | {'No Consensus':<20} | 0/{total}      | -")
    print("")

GENE       | GROUP | CONSENSUS MATCH (m/z) | STABILITY  | STATUS
---------------------------------------------------------------------------
--- AC149090.1 ---
           | YC    | 770.507              | 2/4       | MODERATE
           | YAD   | 846.5419             | 3/4       | STRONG
           | AC    | 868.6625             | 2/4       | MODERATE
           | AAD   | 828.552              | 3/4       | STRONG

--- Aplp1 ---
           | YC    | 841.6462             | 2/4       | MODERATE
           | YAD   | 628.5358             | 3/4       | STRONG
           | AC    | 821.5267             | 1/4       | WEAK
           | AAD   | 867.6604             | 2/4       | MODERATE

--- Apoe ---
           | YC    | 840.6411             | 2/4       | MODERATE
           | YAD   | 524.3706             | 2/4       | MODERATE
           | AC    | 753.5863             | 1/4       | WEAK
           | AAD   | 729.5904             | 1/4       | WEAK

--- App ---
           | YC    | 769.592        

In [12]:
import pandas as pd
import numpy as np

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_FILE = '20_gene_to_mz_results_v1_analytic_fast/gene_to_mz_top30_matches_all_scores.csv' 
K_FACTOR = 100
FILTER_GROUP = None
TOLERANCE = 0.015  # Mass tolerance window (Daltons)

# --- Mass Differences (Positive Mode) ---
# These are the differences between a "Parent" [M+H] and its "Children"
DIFFS = {
    'Isotope (M+1)': 1.003355,   # 13C
    'Isotope (M+2)': 2.006710,   # 13C * 2
    'Adduct (NH4)':  17.02655,   # [M+NH4]+ vs [M+H]+
    'Adduct (Na)':   21.98204,   # [M+Na]+  vs [M+H]+
    'Adduct (K)':    37.95548,   # [M+K]+   vs [M+H]+
    # Note: We treat Water Loss as a child too (if Parent is M+H, child is M+H-H2O)
    'Loss (H2O)':    -18.01056   
}

# =============================================================================
# HELPER: CHECK RELATIONSHIPS
# =============================================================================
def get_relationship(parent_mz, child_mz):
    """
    Returns the relationship name if 'child_mz' is a variant of 'parent_mz'.
    Returns None if no relationship is found.
    """
    diff = child_mz - parent_mz
    
    # Check absolute differences
    if abs(diff - DIFFS['Isotope (M+1)']) < TOLERANCE: return "M+1"
    if abs(diff - DIFFS['Isotope (M+2)']) < TOLERANCE: return "M+2"
    if abs(diff - DIFFS['Adduct (NH4)']) < TOLERANCE: return "NH4"
    if abs(diff - DIFFS['Adduct (Na)']) < TOLERANCE: return "Na"
    if abs(diff - DIFFS['Adduct (K)']) < TOLERANCE: return "K"
    if abs(diff - DIFFS['Loss (H2O)']) < TOLERANCE: return "H2O Loss"
    
    # Check Sodium vs Ammonium (Common case where both exist but no M+H)
    # If Parent is NH4, Child might be Na. Diff = Na - NH4 = ~4.95
    na_nh4_diff = DIFFS['Adduct (Na)'] - DIFFS['Adduct (NH4)']
    if abs(diff - na_nh4_diff) < TOLERANCE: return "Na (vs NH4)"

    return None

# =============================================================================
# CONSENSUS ALGORITHM
# =============================================================================
def calculate_consensus(df, gene_name):
    subset = df[df['Gene'] == gene_name]
    if subset.empty: return None
    
    total_samples_in_study = subset['Sample'].nunique()
    
    # 1. First Pass: Gather raw data per m/z
    # ------------------------------------------------
    mz_data = {} # {mz: {'samples': set(), 'rrf': 0, 'ranks': []}}
    
    for _, row in subset.iterrows():
        mz = row['MZ_Feature']
        sample = row['Sample']
        rank = row['Rank']
        
        if mz not in mz_data:
            mz_data[mz] = {'samples': set(), 'rrf': 0.0, 'min_rank': 99}
            
        mz_data[mz]['samples'].add(sample)
        mz_data[mz]['rrf'] += 1.0 / (rank + K_FACTOR)
        if rank < mz_data[mz]['min_rank']:
            mz_data[mz]['min_rank'] = rank

    if not mz_data: return None

    # 2. Second Pass: Unify Counts (Parent absorbs Children)
    # ------------------------------------------------
    # For every MZ (potential parent), look at every other MZ.
    # If the other MZ is an isotope/adduct, add its samples to the parent.
    
    candidates = list(mz_data.keys())
    
    final_stats = []
    
    for parent in candidates:
        # Start with the samples where the parent ITSELF was found
        consolidated_samples = mz_data[parent]['samples'].copy()
        children_found = []
        
        for child in candidates:
            if parent == child: continue
            
            rel = get_relationship(parent, child)
            if rel:
                # The 'child' is a variant of 'parent'. 
                # Add child's samples to parent's count.
                consolidated_samples.update(mz_data[child]['samples'])
                children_found.append(f"{rel} ({child:.4f})")
        
        count = len(consolidated_samples)
        
        final_stats.append({
            'mz': parent,
            'rrf': mz_data[parent]['rrf'],
            'count': count, # Boosted count
            'raw_count': len(mz_data[parent]['samples']),
            'children': "; ".join(children_found)
        })

    # 3. Sort by RRF Score (primary) and Count (secondary)
    # ------------------------------------------------
    # Note: You might want to sort by Count now that it's boosted, 
    # but usually keeping RRF as primary sorter is safer for relevance.
    final_stats.sort(key=lambda x: x['rrf'], reverse=True)
    
    winner = final_stats[0]
    
    # Check if the winner is ITSELF a child of someone else (e.g. if winner is Na adduct)
    # This is for the "Warning" flag, separate from counting.
    chem_flag = "Primary"
    for other in candidates:
        if other == winner['mz']: continue
        rel = get_relationship(other, winner['mz']) # Is Winner a child of Other?
        if rel:
            chem_flag = f"{rel} of {other:.4f}"
            break

    return {
        'winner': winner['mz'],
        'score': winner['rrf'],
        'count': winner['count'],
        'raw_count': winner['raw_count'],
        'total_samples': total_samples_in_study,
        'chem_flag': chem_flag,
        'merged_notes': winner['children']
    }

# =============================================================================
# MAIN EXECUTION
# =============================================================================
try:
    df = pd.read_csv(INPUT_FILE)
    df.columns = [c.title() for c in df.columns]
    if 'Rna_Sample' in df.columns: df.rename(columns={'Rna_Sample': 'Sample'}, inplace=True)
    if 'Mz_Feature' in df.columns: df.rename(columns={'Mz_Feature': 'MZ_Feature'}, inplace=True)
except FileNotFoundError:
    print(f"Error: Could not find '{INPUT_FILE}'.")
    exit()

if FILTER_GROUP:
    df = df[df['Sample'].str.contains(FILTER_GROUP)]

unique_genes = df['Gene'].unique()

print(f"\n{'GENE':<15} | {'WINNER':<10} | {'SAMPLES':<10} | {'CONFIDENCE':<10} | {'NOTES'}")
print("-" * 115)

results = []

for gene in unique_genes:
    res = calculate_consensus(df, gene)
    if not res: continue
    
    winner = res['winner']
    pct = (res['count'] / res['total_samples']) * 100
    
    confidence = "LOW"
    if pct >= 50: confidence = "MEDIUM"
    if pct >= 75: confidence = "HIGH"
    if pct >= 90: confidence = "VERY HIGH"
    
    # Format sample string (Boosted vs Raw)
    # e.g. "12/12" (if boosted) or "12/12 (Raw: 8)"
    freq_str = f"{res['count']}/{res['total_samples']}"
    
    # Notes display
    notes = []
    if res['chem_flag'] != "Primary":
        notes.append(f"⚠️ {res['chem_flag']}")
    if res['merged_notes']:
        notes.append(f"Incl: {res['merged_notes']}")
    
    note_str = " | ".join(notes)

    print(f"{gene:<15} | {winner:<10.4f} | {freq_str:<10} | {confidence:<10} | {note_str}")
    
    results.append({
        'Gene': gene,
        'Consensus_MZ': winner,
        'Boosted_Count': res['count'],
        'Raw_Count': res['raw_count'],
        'Total_Samples': res['total_samples'],
        'Confidence': confidence,
        'Is_Isotope_Flag': res['chem_flag'],
        'Merged_Isotopes': res['merged_notes']
    })

pd.DataFrame(results).to_csv('gene_to_mz_results_v1_analytic_fast/final_gene_mz_consensus_boosted.csv', index=False)
print("-" * 115)
print("Consensus list saved. Counts now include isotopes/adducts found in other samples.")


GENE            | WINNER     | SAMPLES    | CONFIDENCE | NOTES
-------------------------------------------------------------------------------------------------------------------
AC149090.1      | 770.5070   | 8/16       | MEDIUM     | Incl: M+2 (772.5245); M+2 (772.5258)
Aplp1           | 719.5768   | 6/16       | LOW        | ⚠️ M+1 of 718.5737
Apoe            | 719.5768   | 4/16       | LOW        | ⚠️ M+1 of 718.5737 | Incl: M+1 (720.5885); M+2 (721.5908)
App             | 754.5364   | 8/16       | MEDIUM     | ⚠️ Na of 732.5530 | Incl: M+1 (755.5368); M+2 (756.5500); M+1 (755.5383); M+2 (756.5514); M+1 (755.5369); M+2 (756.5515)
Atp1a3          | 630.6164   | 5/16       | LOW        | 
Elavl3          | 772.6211   | 6/16       | LOW        | Incl: Na (794.6001)
Eno1            | 844.6774   | 5/16       | LOW        | ⚠️ M+2 of 842.6611 | Incl: Na (866.6588)
Gnas            | 754.5364   | 9/16       | MEDIUM     | ⚠️ Na (vs NH4) of 749.5893 | Incl: M+1 (755.5368); M+2 (756.5500); 

In [ ]:
import pandas as pd
import numpy as np

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_FILE = '30_gene_to_mz_synced_results_v1_analytic_fast/gene_to_mz_top30_matches_all_scores.csv' 

# RRF Constant (k): 
# k=0 makes Rank 1 much more valuable than Rank 2 (aggressive).
# k=60 makes Rank 1 and Rank 10 closer in value (smoother).
# For Top 10 lists, k=1 is usually a good balance.
K_FACTOR = 1

# Updated Group List
GROUPS = ['AAD', 'YC', 'AC', 'YAD'] 

TOLERANCE = 0.015  

# --- Mass Differences (Positive Mode) ---
DIFFS = {
    'Isotope (M+1)': 1.003355,   
    'Isotope (M+2)': 2.006710,   
    'Adduct (NH4)':  17.02655,   
    'Adduct (Na)':   21.98204,   
    'Adduct (K)':    37.95548,   
    #'Loss (H2O)':    -18.01056   
}

# =============================================================================
# HELPER: CHECK RELATIONSHIPS
# =============================================================================
def get_relationship(parent_mz, child_mz):
    diff = child_mz - parent_mz
    if abs(diff - DIFFS['Isotope (M+1)']) < TOLERANCE: return "M+1"
    if abs(diff - DIFFS['Isotope (M+2)']) < TOLERANCE: return "M+2"
    if abs(diff - DIFFS['Adduct (NH4)']) < TOLERANCE: return "NH4"
    if abs(diff - DIFFS['Adduct (Na)']) < TOLERANCE: return "Na"
    if abs(diff - DIFFS['Adduct (K)']) < TOLERANCE: return "K"
    if abs(diff - DIFFS['Loss (H2O)']) < TOLERANCE: return "H2O Loss"
    
    # Sodium vs Ammonium check
    na_nh4_diff = DIFFS['Adduct (Na)'] - DIFFS['Adduct (NH4)']
    if abs(diff - na_nh4_diff) < TOLERANCE: return "Na (vs NH4)"
    return None

def identify_group(sample_name):
    """Returns the group name if found in sample_name, else 'Other'"""
    for g in GROUPS:
        if g in sample_name:
            return g
    return "Other"

# =============================================================================
# CONSENSUS ALGORITHM
# =============================================================================
def calculate_consensus(df, gene_name):
    subset = df[df['Gene'] == gene_name]
    if subset.empty: return None
    
    # Calculate Total Samples per Group (for denominator)
    all_samples = subset['Sample'].unique()
    total_samples_overall = len(all_samples)
    
    total_counts_per_group = {g: 0 for g in GROUPS + ['Other']}
    for s in all_samples:
        total_counts_per_group[identify_group(s)] += 1
    
    # 1. First Pass: Gather raw data per m/z
    mz_data = {} 
    
    for _, row in subset.iterrows():
        mz = row['MZ_Feature']
        sample = row['Sample']
        rank = row['Rank']
        group = identify_group(sample)
        
        if mz not in mz_data:
            mz_data[mz] = {
                'samples': set(), 
                'group_samples': {g: set() for g in GROUPS + ['Other']},
                'rrf': 0.0
            }
            
        mz_data[mz]['samples'].add(sample)
        mz_data[mz]['group_samples'][group].add(sample)
        mz_data[mz]['rrf'] += 1.0 / (rank + K_FACTOR)

    if not mz_data: return None

    # 2. Second Pass: Unify Counts (Parent absorbs Children)
    candidates = list(mz_data.keys())
    final_stats = []
    
    for parent in candidates:
        # Clone sets
        consolidated_all = mz_data[parent]['samples'].copy()
        consolidated_groups = {g: mz_data[parent]['group_samples'][g].copy() for g in GROUPS + ['Other']}
        
        children_found = []
        
        for child in candidates:
            if parent == child: continue
            
            rel = get_relationship(parent, child)
            if rel:
                # Merge ALL samples
                consolidated_all.update(mz_data[child]['samples'])
                
                # Merge GROUP samples
                for g in GROUPS + ['Other']:
                    consolidated_groups[g].update(mz_data[child]['group_samples'][g])
                
                children_found.append(f"{rel}")

        children_found = list(set(children_found))

        # Build Group Strings (e.g., "AAD:3/4")
        group_str_parts = []
        group_counts_numeric = {}
        
        for g in GROUPS:
            found = len(consolidated_groups[g])
            total = total_counts_per_group[g]
            group_str_parts.append(f"{g}:{found}/{total}")
            group_counts_numeric[g] = found

        final_stats.append({
            'mz': parent,
            'rrf': mz_data[parent]['rrf'],
            'total_count': len(consolidated_all),
            'group_counts': group_counts_numeric,
            'group_display': " | ".join(group_str_parts),
            'children': ", ".join(children_found)
        })

    # 3. Sort
    final_stats.sort(key=lambda x: x['rrf'], reverse=True)
    winner = final_stats[0]
    
    # Check if winner is potentially an artifact itself
    chem_flag = "Primary"
    for other in candidates:
        if other == winner['mz']: continue
        rel = get_relationship(other, winner['mz'])
        if rel:
            chem_flag = f"{rel} of {other:.4f}"
            break

    return {
        'winner': winner['mz'],
        'data': winner,
        'chem_flag': chem_flag,
        'total_samples_overall': total_samples_overall
    }

# =============================================================================
# MAIN EXECUTION
# =============================================================================
try:
    df = pd.read_csv(INPUT_FILE)
    df.columns = [c.title() for c in df.columns]
    if 'Rna_Sample' in df.columns: df.rename(columns={'Rna_Sample': 'Sample'}, inplace=True)
    if 'Mz_Feature' in df.columns: df.rename(columns={'Mz_Feature': 'MZ_Feature'}, inplace=True)
except FileNotFoundError:
    print(f"Error: Could not find '{INPUT_FILE}'.")
    exit()

unique_genes = df['Gene'].unique()

print(f"\n{'GENE':<15} | {'WINNER':<10} | {'TOTAL':<8} | {'CONF':<8} | {'GROUP BREAKDOWN':<45} | {'NOTES'}")
print("-" * 140)

results = []

for gene in unique_genes:
    res = calculate_consensus(df, gene)
    if not res: continue
    
    w = res['data']
    total_found = w['total_count']
    total_possible = res['total_samples_overall']
    
    # Calculate Confidence
    pct = (total_found / total_possible) * 100
    confidence = "LOW"
    if pct >= 50: confidence = "MEDIUM"
    if pct >= 75: confidence = "HIGH"
    if pct >= 90: confidence = "V.HIGH"
    
    total_str = f"{total_found}/{total_possible}"

    # Notes
    notes = []
    if res['chem_flag'] != "Primary": notes.append(f"⚠️ {res['chem_flag']}")
    if w['children']: notes.append(f"Incl: {w['children']}")
    note_str = "; ".join(notes)

    print(f"{gene:<15} | {res['winner']:<10.4f} | {total_str:<8} | {confidence:<8} | {w['group_display']:<45} | {note_str}")
    
    # Save Dict
    row_data = {
        'Gene': gene,
        'Consensus_MZ': res['winner'],
        'Total_Found': total_found,
        'Total_Possible': total_possible,
        'Confidence': confidence,
        'Isotope_Flag': res['chem_flag'],
        'Merged_Isotopes': w['children']
    }
    # Add dynamic group columns
    for g in GROUPS:
        row_data[f'Count_{g}'] = w['group_counts'][g]
        
    results.append(row_data)

# Save
pd.DataFrame(results).to_csv('30_gene_to_mz_synced_results_v1_analytic_fast/final_gene_to_mz_top30_matches_all_scores.csv', index=False)
print("-" * 140)
print("Complete consensus list saved.")


GENE            | WINNER     | TOTAL    | CONF     | GROUP BREAKDOWN                               | NOTES
--------------------------------------------------------------------------------------------------------------------------------------------
AC149090.1      | 821.5308   | 9/16     | MEDIUM   | AAD:0/4 | YC:3/4 | AC:3/4 | YAD:3/4           | ⚠️ M+1 of 820.5250
Aplp1           | 628.5358   | 6/16     | LOW      | AAD:2/4 | YC:0/4 | AC:1/4 | YAD:3/4           | ⚠️ M+1 of 627.5340; Incl: M+1
Apoe            | 780.5501   | 6/16     | LOW      | AAD:0/4 | YC:4/4 | AC:1/4 | YAD:1/4           | ⚠️ Na of 758.5688; Incl: M+1
App             | 755.5406   | 10/16    | MEDIUM   | AAD:3/4 | YC:4/4 | AC:2/4 | YAD:1/4           | ⚠️ M+1 of 754.5364; Incl: M+1, M+2
Atp1a3          | 878.5697   | 9/16     | MEDIUM   | AAD:3/4 | YC:1/4 | AC:1/4 | YAD:4/4           | ⚠️ Na of 856.5819
Elavl3          | 858.6960   | 10/16    | MEDIUM   | AAD:4/4 | YC:0/4 | AC:2/4 | YAD:4/4           | Incl: Na, M+1
